# Visualising Datasets in 2D with NCD + MDS

This notebook embeds each HTML dataset into a 2D plane so that **structurally
similar pages land near each other**, with no clustering involved.

The pipeline is:

1. **Distance** — for every pair of pages we compute the
   [Normalized Compression Distance](../../AGENTS.md) (NCD) using the
   `ncd` Rust extension (Python bindings for the `kolmox` crate):

   $$ d(P_A, P_B) = \frac{C(P_A P_B) - \min(C(P_A), C(P_B))}{\max(C(P_A), C(P_B))}, \quad 0 \le d \le 1 $$

   where $C(\cdot)$ is the **zstd**-compressed size (level 18, matching the Rust
   benchmarks' `recommended()` settings). Pages are reduced to their structural
   form (tags + `id`/`class`, no text) before compression.

2. **Embedding** — we feed the resulting distance matrix to
   [Multidimensional Scaling](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.MDS.html)
   (MDS) with `dissimilarity="precomputed"`, which finds 2D coordinates whose
   pairwise Euclidean distances best preserve the NCD distances.

3. **Plot** — a 2D scatter, coloured by the page *type* recorded in the
   dataset, so we can see whether structurally similar pages share a type.

In [6]:
import warnings

import ncd

warnings.filterwarnings("ignore", module="sklearn")  # quiet MDS convergence chatter
print("ncd extension loaded:", [f for f in dir(ncd) if not f.startswith("_")])

ncd extension loaded: ['distance_matrix', 'ncd', 'page_distance']


## Loading a dataset

Each dataset is a directory under [`dataset/`](../../dataset) containing a
`dataset.csv` (`url, type`) plus the saved HTML files. The helpers below mirror
the path-resolution logic in the Rust `dataset` module: a URL is turned into a
file path by stripping the scheme and domain, URL-decoding the rest, and trying
a few extensions (`.html`, `.txt`, `.md`, `.wiki`, none) or a directory
`index.html`.

In [7]:
from pathlib import Path
from urllib.parse import unquote
import pandas as pd

DATASET_ROOT = Path("../../dataset").resolve()


def url_to_filename(url: str) -> str | None:
    i = url.find("://")
    if i == -1:
        return None
    rest = url[i + 3 :]
    j = rest.find("/")
    if j == -1:
        return None
    return unquote(rest[j + 1 :])


def resolve_path(directory: Path, url: str) -> Path | None:
    name = url_to_filename(url)
    if name is None:
        return None
    for ext in (".html", ".txt", ".md", ".wiki", ""):
        p = directory / f"{name}{ext}"
        if p.is_dir():
            idx = p / "index.html"
            if idx.exists():
                return idx
        if p.exists() and p.is_file():
            return p
    return None


def short_label(url: str, max_part: int = 12) -> str:
    """A compact, human-readable label: the URL path with long parts trimmed."""
    name = url_to_filename(url) or url
    parts = [p if len(p) <= max_part else p[:max_part] + "…" for p in name.split("/")]
    return "/" + "/".join(parts)


def load_dataset(name: str) -> pd.DataFrame:
    directory = DATASET_ROOT / name
    df = pd.read_csv(directory / "dataset.csv", skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    df["url"] = df["url"].str.strip().str.strip('"')
    df["type"] = df["type"].str.strip().str.strip('"')
    df["path"] = df["url"].map(lambda u: resolve_path(directory, u))
    df = df[df["path"].notna()].reset_index(drop=True)
    df["html"] = df["path"].map(lambda p: p.read_text(errors="replace"))
    df["label"] = df["url"].map(short_label)
    return df


df = load_dataset("euronews.com")
print(f"{len(df)} pages, types: {df['type'].value_counts().to_dict()}")
df[["label", "type"]].head()

26 pages, types: {'article': 6, 'profile': 6, 'tag': 6, 'section': 4, 'special': 4}


,label,type
0,/2025/08/16/zelenskyy-an…,article
1,/my-europe/2025/08/16/european-lea…,article
2,/2025/08/16/death-toll-r…,article
3,/2025/08/16/uk-trade-env…,article
4,/2025/08/16/protesters-o…,article


## Distances → 2D coordinates

`ncd.distance_matrix` computes the full pairwise NCD matrix in parallel in Rust
(with a shared compression cache). We symmetrise it defensively and hand it to
MDS to obtain 2D coordinates.

We use **metric** MDS (`metric=True`): it preserves the *distances* of the
NCD distances rather than just their ordering, which suits a qualitative 2D
map.

In [ ]:
import numpy as np
from sklearn.manifold import MDS


def embed_2d(htmls: list[str], quality: int = 18) -> np.ndarray:
    matrix = np.array(ncd.distance_matrix(htmls, quality=quality))
    #matrix = (matrix + matrix.T) / 2.0  # enforce exact symmetry for MDS
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        metric=True, #False,
        random_state=42,
        normalized_stress="auto",
        n_init=8,
        max_iter=600,
        eps=1e-4,
    )
    coords = mds.fit_transform(matrix)
    return coords, mds.stress_


coords, stress = embed_2d(df["html"].tolist())
print("MDS stress:", round(stress, 4))
coords[:5]

MDS stress: 0.7741


array([[-0.19503054,  0.18630044],
       [-0.1781336 ,  0.16805178],
       [-0.19368534,  0.12604192],
       [-0.18480078,  0.10309847],
       [-0.18828555,  0.12647607]])

## Plotting

Each point is a page, positioned by its MDS coordinates and coloured by type.
Hover to see the page path. Structurally similar pages cluster together
*visually* — but note we are **not** running any clustering algorithm, just
showing the raw 2D embedding.

In [9]:
import plotly.express as px


def plot_dataset(name: str, quality: int = 18):
    data = load_dataset(name)
    coords, stress = embed_2d(data["html"].tolist(), quality=quality)
    data = data.assign(x=coords[:, 0], y=coords[:, 1])
    fig = px.scatter(
        data,
        x="x",
        y="y",
        color="type",
        hover_name="label",
        hover_data={"x": False, "y": False, "url": True, "type": True},
        title=f"NCD + MDS 2D embedding — {name} ({len(data)} pages, stress={stress:.3f})",
    )
    fig.update_traces(marker=dict(size=11, line=dict(width=1, color="white")))
    fig.update_layout(width=850, height=650, xaxis_title="MDS-1", yaxis_title="MDS-2")
    return fig


plot_dataset("euronews.com").show()

## Other datasets

The same pipeline applied to the remaining datasets. Datasets with a single
page type (e.g. all `article` or all `movie`) still spread out by structural
variation; mixed datasets like `wikipedia` show types separating in space.

In [10]:
for dataset_name in ["imdb", "amazon", "wikipedia"]:
    plot_dataset(dataset_name).show()